In [6]:
pip install ipykernel pandas tabulate


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
con = sqlite3.connect(ROOT / "db" / "weblog.db")

# 1. 행 수
print(pd.read_sql("SELECT COUNT(*) AS n FROM events", con))

# 2. 기간
print(pd.read_sql("SELECT MIN(event_time) s, MAX(event_time) e FROM events", con))

# 3. 이벤트 종류
print(pd.read_sql("SELECT event_type, COUNT(*) c FROM events GROUP BY 1 ORDER BY c DESC", con))

# 4. 결측률
print(pd.read_sql("""
SELECT ROUND(AVG(category_code IS NULL)*100, 1) cat_null_pct,
       ROUND(AVG(brand IS NULL)*100, 1) brand_null_pct
FROM events
""", con))

# 5. 규모 감각
print(pd.read_sql("""
SELECT COUNT(DISTINCT user_id) users,
       COUNT(DISTINCT user_session) sessions,
       COUNT(DISTINCT product_id) products
FROM events
""", con))

         n
0  6610750
                     s                    e
0  2019-10-01 00:00:00  2019-10-31 23:59:58
  event_type        c
0       view  6379532
1       cart   124252
2   purchase   106966
   cat_null_pct  brand_null_pct
0          32.6            16.2
   users  sessions  products
0  91653    935455    132720


In [10]:
con = sqlite3.connect("../db/weblog.db")

q1 = "SELECT MIN(c) min_ev, MAX(c) max_ev, AVG(c) avg_ev FROM (SELECT COUNT(*) c FROM events GROUP BY user_id)"
q2 = "SELECT COUNT(*) single_event_users FROM (SELECT user_id FROM events GROUP BY user_id HAVING COUNT(*) = 1)"
q3 = "SELECT event_type, COUNT(*) n FROM events GROUP BY event_type ORDER BY n DESC"
q4 = "SELECT event_time, strftime('%w', event_time) dow FROM events LIMIT 1"

for name, q in [("유저별 이벤트 수", q1), ("1건 유저", q2), ("event_type", q3), ("strftime 확인", q4)]:
    print(f"\n## {name}")
    print(pd.read_sql(q, con).to_markdown(index=False))
print("ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ")
cnt = pd.read_sql("SELECT user_id, COUNT(*) n FROM events GROUP BY user_id", con)
print(cnt.n.describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_markdown())

con.close()


## 유저별 이벤트 수
|   min_ev |   max_ev |   avg_ev |
|---------:|---------:|---------:|
|        1 |     7436 |   72.128 |

## 1건 유저
|   single_event_users |
|---------------------:|
|                 1674 |

## event_type
| event_type   |       n |
|:-------------|--------:|
| view         | 6379532 |
| cart         |  124252 |
| purchase     |  106966 |

## strftime 확인
| event_time          |   dow |
|:--------------------|------:|
| 2019-10-01 00:00:00 |     2 |
ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ
|       |         n |
|:------|----------:|
| count | 91653     |
| mean  |    72.128 |
| std   |   113.743 |
| min   |     1     |
| 25%   |    15     |
| 50%   |    37     |
| 75%   |    86     |
| 90%   |   170     |
| 95%   |   252     |
| 99%   |   521     |
| max   |  7436     |


In [2]:
import sqlite3, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "db" / "weblog.db"

def q(sql, md=True):
    with sqlite3.connect(DB) as con:
        df = pd.read_sql(sql, con)
    print(df.to_markdown(index=False)) if md else None
    return df

In [3]:
q("""
WITH cart AS (
  SELECT user_id, product_id, MIN(event_time) AS first_cart
  FROM events
  WHERE event_type = 'cart'
  GROUP BY user_id, product_id
)
SELECT COUNT(*) AS purchase_total,
       SUM(CASE WHEN c.first_cart <= p.event_time THEN 1 ELSE 0 END) AS with_cart_any_time
FROM events p
LEFT JOIN cart c
  ON c.user_id = p.user_id AND c.product_id = p.product_id
WHERE p.event_type = 'purchase'
""")

|   purchase_total |   with_cart_any_time |
|-----------------:|---------------------:|
|           106966 |                53387 |


,purchase_total,with_cart_any_time
0,106966,53387


In [4]:
q("""
SELECT ROUND(MIN(price),2) mn, ROUND(MAX(price),2) mx,
       ROUND(AVG(price),2) avg,
       COUNT(CASE WHEN price IS NULL OR price <= 0 THEN 1 END) bad
FROM events WHERE event_type='purchase'
""")
q("SELECT price FROM events WHERE event_type='purchase' ORDER BY price").price.describe(percentiles=[.25,.5,.75,.9])

|   mn |      mx |    avg |   bad |
|-----:|--------:|-------:|------:|
| 0.77 | 2574.04 | 311.29 |     0 |
|   price |
|--------:|
|    0.77 |
|    0.79 |
|    0.79 |
|    0.83 |
|    0.83 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.85 |
|    0.88 |
|    0.88 |
|    0.88 |
|    0.88 |
|    0.88 |
|    0.88 |
|    0.9  |
|    0.9  |
|    0.92 |
|    0.92 |
|    0.99 |
|    0.99 |
|    1    |
|    1.01 |
|    1.01 |
|    1.09 |
|    1.09 |
|    1.1  |
|    1.12 |
|    1.13 |
|    1.14 |
|    1.24 |
|    1.24 |
|    1.25 |
|    1.25 |
|    1.26 |
|    1.26 |
|    1.26 |
|    1.26 |
|    1.26 |
|    1.26 |
|    1.26 |
|    1.29 |
|    1.29 |
|    1.29 |
|    1.31 |
|    1.31 |
|    1.31 |
|    1.38 |
|    1.39 |
|    1.39 |
|    1.39 |
|    1.39 |
|    1.4  |
|    1.4  |
|    1.4  |
|    1.42 |
|    1.43 |
|    1.43 |
|    1.44 |
|    1.45 |
|    1.52 |
|    1.52 |
|    1.52 |
|    1.52 |
|    1.52 |
|    1.53 |
|    1.53 |
|   

count    106966.000000
mean        311.288051
std         351.479588
min           0.770000
25%          82.370000
50%         180.160000
75%         380.912500
90%         795.840000
max        2574.040000
Name: price, dtype: float64